1. Synthetic Data Generator (Schema-Based)

In [2]:
"""
synthetic_orchestrator_data.py
Generates training data with dynamic model schemas
"""
import json
import random
import uuid
from typing import List, Dict, Any
from datasets import Dataset

def generate_random_schema():
    """Generate random input/output schemas"""
    types = ["string", "integer", "float", "boolean", "enum"]
    fields = {}
    for i in range(random.randint(1, 4)):
        field_type = random.choice(types)
        if field_type == "enum":
            values = random.sample(["A", "B", "C", "D", "auto", "fast", "slow"], 2)
            fields[f"param_{i}"] = {"type": "enum", "values": values}
        else:
            fields[f"param_{i}"] = {"type": field_type}
    return fields

def generate_model_card():
    """Generate a random model metadata card"""
    domains = ["vision", "nlp", "math", "search", "generation", "analysis"]
    actions = ["extract", "transform", "generate", "classify", "retrieve", "calculate"]
    
    model_id = f"{random.choice(actions)}_{random.choice(domains)}_{uuid.uuid4().hex[:4]}"
    
    return {
        "model_id": model_id,
        "description": f"A model that {random.choice(actions)}s {random.choice(domains)} data",
        "capabilities": random.sample(domains, k=random.randint(1, 2)),
        "input_schema": generate_random_schema(),
        "output_schema": generate_random_schema(),
        "latency": random.choice(["low", "medium", "high"]),
        "cost_tier": random.choice(["free", "standard", "premium"])
    }

def generate_query_for_models(models: List[Dict], num_calls: int) -> tuple:
    """
    Generate a natural query and expected execution plan
    Returns: (query, execution_plan)
    """
    selected = random.sample(models, min(num_calls, len(models)))
    
    # Create query by combining model descriptions
    if num_calls == 1:
        templates = [
            "Use {model_desc} to process {input_desc}",
            "Run {model_id} on {input_desc}",
            "{task} using {model_desc}"
        ]
    else:
        templates = [
            "{task1} and also {task2}",
            "First {task1}, then {task2}",
            "{task1} and {task2} in parallel",
            "Compare {task1} with {task2}"
        ]
    
    tasks = []
    calls = []
    
    for i, model in enumerate(selected):
        # Generate valid inputs based on schema
        inputs = {}
        for param, spec in model["input_schema"].items():
            if spec["type"] == "string":
                inputs[param] = random.choice(["data.txt", "query", "input_data"])
            elif spec["type"] == "integer":
                inputs[param] = random.randint(1, 100)
            elif spec["type"] == "float":
                inputs[param] = round(random.uniform(0.1, 10.0), 2)
            elif spec["type"] == "boolean":
                inputs[param] = random.choice([True, False])
            elif spec["type"] == "enum":
                inputs[param] = random.choice(spec["values"])
        
        calls.append({
            "model_id": model["model_id"],
            "arguments": inputs,
            "step": i + 1
        })
        
        tasks.append(f"use {model['model_id']} with {list(inputs.keys())}")
    
    # Build query
    if num_calls == 1:
        query = random.choice(templates).format(
            model_desc=selected[0]["description"],
            model_id=selected[0]["model_id"],
            input_desc=list(selected[0]["input_schema"].keys())[0],
            task=tasks[0]
        )
    else:
        query = random.choice(templates).format(
            task1=tasks[0],
            task2=tasks[1] if len(tasks) > 1 else tasks[0]
        )
    
    # Determine dependencies (parallel vs sequential)
    if random.random() > 0.3:  # 70% parallel for independent calls
        execution_plan = {
            "plan_type": "parallel",
            "steps": [{
                "step_id": 1,
                "parallel_calls": calls,
                "dependencies": []
            }]
        }
    else:  # Sequential with dependencies
        execution_plan = {
            "plan_type": "sequential", 
            "steps": []
        }
        for i, call in enumerate(calls):
            execution_plan["steps"].append({
                "step_id": i + 1,
                "call": call,
                "dependencies": [i] if i > 0 else []
            })
    
    return query, execution_plan

def generate_orchestrator_dataset(num_samples: int = 5000):
    """Generate full training dataset"""
    dataset = []
    
    for _ in range(num_samples):
        # Generate 3-8 available models (simulates dynamic registry)
        available_models = [generate_model_card() for _ in range(random.randint(3, 8))]
        
        # 1-3 function calls per query
        num_calls = random.choices([1, 2, 3], weights=[0.5, 0.35, 0.15])[0]
        
        query, plan = generate_query_for_models(available_models, num_calls)
        
        # Format as chat with system context containing metadata
        system_msg = {
            "role": "system",
            "content": f"""You are an AI orchestrator. Select and chain models based on their metadata to fulfill the user request.

Available Models:
{json.dumps(available_models, indent=2)}

Rules:
1. Reference models by their exact model_id
2. Arguments must match the input_schema types exactly
3. Set dependencies when a step requires output from previous steps
4. Parallel calls should be in the same step with empty dependencies"""
        }
        
        example = {
            "messages": [
                system_msg,
                {"role": "user", "content": query},
                {"role": "assistant", "content": json.dumps(plan)}
            ],
            "metadata": {
                "available_models": [m["model_id"] for m in available_models],
                "num_calls": num_calls
            }
        }
        dataset.append(example)
    
    return Dataset.from_list(dataset)

if __name__ == "__main__":
    ds = generate_orchestrator_dataset(1000)
    print(f"Generated {len(ds)} examples")
    print("\nSample:")
    print(json.dumps(ds[0]["messages"], indent=2))

Generated 1000 examples

Sample:
[
  {
    "content": "You are an AI orchestrator. Select and chain models based on their metadata to fulfill the user request.\n\nAvailable Models:\n[\n  {\n    \"model_id\": \"extract_math_489c\",\n    \"description\": \"A model that generates vision data\",\n    \"capabilities\": [\n      \"analysis\"\n    ],\n    \"input_schema\": {\n      \"param_0\": {\n        \"type\": \"boolean\"\n      }\n    },\n    \"output_schema\": {\n      \"param_0\": {\n        \"type\": \"boolean\"\n      },\n      \"param_1\": {\n        \"type\": \"enum\",\n        \"values\": [\n          \"D\",\n          \"fast\"\n        ]\n      },\n      \"param_2\": {\n        \"type\": \"boolean\"\n      },\n      \"param_3\": {\n        \"type\": \"string\"\n      }\n    },\n    \"latency\": \"medium\",\n    \"cost_tier\": \"standard\"\n  },\n  {\n    \"model_id\": \"transform_analysis_54ca\",\n    \"description\": \"A model that retrieves generation data\",\n    \"capabiliti

2. Multi-GPU Training Script (Kaggle 2x T4)

In [4]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 9.7 MB/s eta 0:00:00a 0:00:01


In [5]:
"""
train_orchestrator_kaggle.py
Optimized for Kaggle's 2x T4 setup using Accelerate
"""
import torch
import json
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from accelerate import Accelerator
from synthetic_orchestrator_data import generate_orchestrator_dataset

# Initialize accelerator for multi-GPU
accelerator = Accelerator()

MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"
OUTPUT_DIR = "./orchestrator-phi3-dynamic"

def formatting_func(example):
    """Format dataset examples using chat template"""
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    text = tokenizer.apply_chat_template(
        example["messages"], 
        tokenize=False, 
        add_generation_prompt=False
    )
    return text

def main():
    # Generate data
    if accelerator.is_main_process:
        print("Generating synthetic dataset...")
    dataset = generate_orchestrator_dataset(5000)
    
    # Split for eval
    dataset = dataset.train_test_split(test_size=0.1)
    
    # Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    
    # Quantization config - T4 optimized (use fp16, not bf16)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,  # T4 uses fp16, not bf16
        bnb_4bit_use_double_quant=True
    )
    
    # Load model
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map={"": accelerator.process_index},  # Let Accelerate handle device placement
        trust_remote_code=True,
        torch_dtype=torch.float16
    )
    
    # Prepare for training
    model = prepare_model_for_kbit_training(model)
    
    # LoRA config - slightly larger for complex orchestration logic
    lora_config = LoraConfig(
        r=32,  # Increased rank for complex schema understanding
        lora_alpha=64,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    
    model = get_peft_model(model, lora_config)
    
    if accelerator.is_main_process:
        model.print_trainable_parameters()
    
    # Training arguments optimized for 2x T4
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=2,  # Reduced for faster iteration
        per_device_train_batch_size=2,  # Per GPU
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,  # Effective batch = 2*2*4 = 16
        learning_rate=1e-4,  # Slightly lower for stability
        fp16=True,  # T4 optimized
        bf16=False,  # T4 doesn't support bf16 well
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=500,
        warmup_ratio=0.1,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        group_by_length=True,  # Efficiency for variable length sequences
        ddp_find_unused_parameters=False,  # Important for DDP
        report_to="none",  # Disable wandb for Kaggle
        load_best_model_at_end=True,
        seed=42
    )
    
    # SFT Trainer handles the formatting and padding
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        args=training_args,
        formatting_func=formatting_func,
        max_seq_length=1024,  # Increased for long metadata context
        packing=False,  # Don't pack - we need clear boundaries for JSON learning
    )
    
    # Train
    trainer.train()
    
    # Save only on main process
    if accelerator.is_main_process:
        trainer.save_model(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        
        # Save adapter config
        model.config.save_pretrained(OUTPUT_DIR)
        print(f"✓ Model saved to {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

ModuleNotFoundError: No module named 'synthetic_orchestrator_data'

In [ ]:
# In a notebook cell:
!accelerate config  # Select multi-GPU, 2 processes, no DeepSpeed
!accelerate launch train_orchestrator_kaggle.py

3. Runtime Orchestrator Engine

In [ ]:
"""
dynamic_orchestrator.py
Runtime system for dynamic model registry and execution
"""
import json
import torch
from typing import List, Dict, Any, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import concurrent.futures
from dataclasses import dataclass

@dataclass
class ModelMetadata:
    model_id: str
    description: str
    input_schema: Dict[str, Any]
    output_schema: Dict[str, Any]
    handler: callable  # Actual function/model to call
    cost_tier: str = "standard"
    latency: str = "medium"

class DynamicOrchestrator:
    def __init__(self, adapter_path: str, base_model: str = "microsoft/Phi-3.5-mini-instruct"):
        self.tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
        
        # Load model
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        
        base = AutoModelForCausalLM.from_pretrained(
            base_model,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )
        
        self.model = PeftModel.from_pretrained(base, adapter_path)
        self.model.eval()
        
        # Dynamic registry
        self.registry: Dict[str, ModelMetadata] = {}
        
    def register_model(self, metadata: ModelMetadata):
        """Add a model to the registry at runtime"""
        self.registry[metadata.model_id] = metadata
        print(f"Registered model: {metadata.model_id}")
        
    def unregister_model(self, model_id: str):
        """Remove a model from the registry"""
        if model_id in self.registry:
            del self.registry[model_id]
            
    def generate_plan(self, query: str) -> Dict[str, Any]:
        """Generate execution plan using current registry"""
        # Build system message with current metadata
        available_models = [
            {
                "model_id": m.model_id,
                "description": m.description,
                "input_schema": m.input_schema,
                "output_schema": m.output_schema,
                "cost_tier": m.cost_tier,
                "latency": m.latency
            }
            for m in self.registry.values()
        ]
        
        system_content = f"""You are an AI orchestrator. Select models from the available registry to fulfill the user request.

Available Models:
{json.dumps(available_models, indent=2)}

Output a JSON execution plan with this exact structure:
{{
  "plan_type": "parallel" | "sequential",
  "steps": [
    {{
      "step_id": int,
      "parallel_calls": [
        {{
          "model_id": "exact_model_id",
          "arguments": {{"param": "value"}},
          "reasoning": "why this model"
        }}
      ],
      "dependencies": [step_ids]
    }}
  ],
  "expected_workflow": "description of the plan"
}}"""

        messages = [
            {"role": "system", "content": system_content},
            {"role": "user", "content": query}
        ]
        
        input_text = self.tokenizer.apply_chat_template(
            messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        
        inputs = self.tokenizer(input_text, return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=800,  # Larger for complex plans
                temperature=0.1,  # Low temp for deterministic JSON
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
                use_cache=False,
                top_p=0.95,
                repetition_penalty=1.1
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        
        # Extract JSON
        try:
            # Find JSON between first { and last }
            start = response.find("{")
            end = response.rfind("}") + 1
            if start != -1 and end != -1:
                plan = json.loads(response[start:end])
                return self.validate_plan(plan)
            else:
                raise ValueError("No JSON found in response")
        except json.JSONDecodeError as e:
            return {
                "error": "Failed to parse plan",
                "raw_response": response,
                "fallback": "single_call"
            }
    
    def validate_plan(self, plan: Dict) -> Dict:
        """Validate that plan only uses registered models"""
        valid_model_ids = set(self.registry.keys())
        
        for step in plan.get("steps", []):
            calls = step.get("parallel_calls", [])
            for call in calls:
                if call["model_id"] not in valid_model_ids:
                    return {
                        "error": f"Model {call['model_id']} not in registry",
                        "plan": plan
                    }
                
                # Validate arguments against schema
                model_meta = self.registry[call["model_id"]]
                schema = model_meta.input_schema
                
                for param, value in call["arguments"].items():
                    if param not in schema:
                        return {
                            "error": f"Invalid param {param} for {call['model_id']}",
                            "plan": plan
                        }
        return plan
    
    def execute_plan(self, plan: Dict) -> Dict[str, Any]:
        """Execute the validated plan"""
        if "error" in plan:
            return plan
            
        results = {"steps": [], "final_outputs": []}
        step_results = {}  # Store results for dependency resolution
        
        for step in plan["steps"]:
            step_id = step["step_id"]
            calls = step["parallel_calls"]
            
            # Check dependencies
            deps_satisfied = all(
                dep in step_results for dep in step["dependencies"]
            )
            if not deps_satisfied:
                return {"error": f"Dependencies not satisfied for step {step_id}"}
            
            # Execute parallel calls
            step_output = {"step_id": step_id, "results": []}
            
            with concurrent.futures.ThreadPoolExecutor(max_workers=len(calls)) as executor:
                future_to_call = {}
                
                for call in calls:
                    model_id = call["model_id"]
                    args = call["arguments"]
                    
                    # Substitute any dependency references (e.g., "$step_1.output")
                    args = self.resolve_dependencies(args, step_results)
                    
                    model_meta = self.registry[model_id]
                    future = executor.submit(model_meta.handler, **args)
                    future_to_call[future] = call
                
                for future in concurrent.futures.as_completed(future_to_call):
                    call = future_to_call[future]
                    try:
                        result = future.result(timeout=30)
                        step_output["results"].append({
                            "model_id": call["model_id"],
                            "input": call["arguments"],
                            "output": result,
                            "status": "success"
                        })
                    except Exception as e:
                        step_output["results"].append({
                            "model_id": call["model_id"],
                            "error": str(e),
                            "status": "failed"
                        })
            
            step_results[step_id] = step_output
            results["steps"].append(step_output)
            
        results["final_outputs"] = self.aggregate_results(results["steps"])
        return results
    
    def resolve_dependencies(self, args: Dict, step_results: Dict) -> Dict:
        """Resolve references like '$step_1.result.field' in arguments"""
        resolved = {}
        for key, value in args.items():
            if isinstance(value, str) and value.startswith("$"):
                # Parse reference path
                parts = value[1:].split(".")
                step_ref = int(parts[0].split("_")[1])
                field_path = parts[1:]
                
                # Navigate to value
                val = step_results.get(step_ref, {})
                for field in field_path:
                    val = val.get(field, {})
                resolved[key] = val
            else:
                resolved[key] = value
        return resolved
    
    def aggregate_results(self, steps: List[Dict]) -> List[Any]:
        """Aggregate final results from last step"""
        if not steps:
            return []
        last_step = steps[-1]
        return [r["output"] for r in last_step["results"] if r["status"] == "success"]
    
    def process(self, query: str) -> Dict:
        """Full pipeline: plan + execute"""
        print(f"Processing: {query}")
        
        # Generate plan
        plan = self.generate_plan(query)
        print(f"Generated plan: {json.dumps(plan, indent=2)}")
        
        # Execute
        results = self.execute_plan(plan)
        return {
            "query": query,
            "plan": plan,
            "execution_results": results
        }

# Example usage
if __name__ == "__main__":
    orch = DynamicOrchestrator("./orchestrator-phi3-dynamic")
    
    # Register models at runtime (these could be loaded from a database/API)
    orch.register_model(ModelMetadata(
        model_id="weather_api",
        description="Retrieves current weather for a location",
        input_schema={"location": "string", "unit": "enum[celsius,fahrenheit]"},
        output_schema={"temp": "float", "condition": "string"},
        handler=lambda location, unit: {"temp": 22.5, "condition": "sunny"},
        latency="low"
    ))
    
    orch.register_model(ModelMetadata(
        model_id="stock_api",
        description="Gets current stock price",
        input_schema={"symbol": "string"},
        output_schema={"price": "float", "currency": "string"},
        handler=lambda symbol: {"price": 150.0, "currency": "USD"},
        latency="medium"
    ))
    
    # Test multi-call
    result = orch.process("What's the weather in Paris and stock price of AAPL?")
    print(json.dumps(result, indent=2))

In [8]:
!pip install -q -U bitsandbytes
!pip install -q transformers accelerate peft trl datasets

In [9]:
# Cell 2: Complete Training Script (Copy-paste this entire cell)
import os
import torch
import json
import random
import uuid
from typing import List, Dict
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# Check available GPUs
print(f"GPUs available: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# ==================== DATA GENERATION ====================
def generate_random_schema():
    """Generate random input/output schemas"""
    types = ["string", "integer", "float", "boolean", "enum"]
    fields = {}
    for i in range(random.randint(1, 4)):
        field_type = random.choice(types)
        if field_type == "enum":
            values = random.sample(["A", "B", "C", "D", "auto", "fast", "slow"], 2)
            fields[f"param_{i}"] = {"type": "enum", "values": values}
        else:
            fields[f"param_{i}"] = {"type": field_type}
    return fields

def generate_model_card():
    """Generate a random model metadata card"""
    domains = ["vision", "nlp", "math", "search", "generation", "analysis"]
    actions = ["extract", "transform", "generate", "classify", "retrieve", "calculate"]
    
    model_id = f"{random.choice(actions)}_{random.choice(domains)}_{uuid.uuid4().hex[:4]}"
    
    return {
        "model_id": model_id,
        "description": f"A model that {random.choice(actions)}s {random.choice(domains)} data",
        "capabilities": random.sample(domains, k=random.randint(1, 2)),
        "input_schema": generate_random_schema(),
        "output_schema": generate_random_schema(),
        "latency": random.choice(["low", "medium", "high"]),
        "cost_tier": random.choice(["free", "standard", "premium"])
    }

def generate_orchestrator_dataset(num_samples: int = 3000):
    """Generate training dataset with dynamic model schemas"""
    dataset = []
    
    for _ in range(num_samples):
        # Generate 3-8 available models (simulates dynamic registry)
        available_models = [generate_model_card() for _ in range(random.randint(3, 8))]
        
        # 1-3 function calls per query
        num_calls = random.choices([1, 2, 3], weights=[0.5, 0.35, 0.15])[0]
        selected = random.sample(available_models, min(num_calls, len(available_models)))
        
        # Generate query and plan
        tasks = []
        calls = []
        
        for i, model in enumerate(selected):
            inputs = {}
            for param, spec in model["input_schema"].items():
                if spec["type"] == "string":
                    inputs[param] = random.choice(["data.txt", "query", "input_data"])
                elif spec["type"] == "integer":
                    inputs[param] = random.randint(1, 100)
                elif spec["type"] == "float":
                    inputs[param] = round(random.uniform(0.1, 10.0), 2)
                elif spec["type"] == "boolean":
                    inputs[param] = random.choice([True, False])
                elif spec["type"] == "enum":
                    inputs[param] = random.choice(spec["values"])
            
            calls.append({
                "model_id": model["model_id"],
                "arguments": inputs,
                "step": i + 1
            })
            
            tasks.append(f"use {model['model_id']}")
        
        # Create query
        if num_calls == 1:
            query = f"Execute {tasks[0]} with parameters {list(inputs.keys())}"
        else:
            templates = [
                f"{tasks[0]} and also {tasks[1]}",
                f"First {tasks[0]}, then {tasks[1]}",
                f"Run {tasks[0]} and {tasks[1]} in parallel",
            ]
            query = random.choice(templates)
        
        # Create execution plan
        if random.random() > 0.3:  # Parallel
            plan = {
                "plan_type": "parallel",
                "steps": [{
                    "step_id": 1,
                    "parallel_calls": calls,
                    "dependencies": []
                }]
            }
        else:  # Sequential
            plan = {
                "plan_type": "sequential", 
                "steps": []
            }
            for i, call in enumerate(calls):
                plan["steps"].append({
                    "step_id": i + 1,
                    "call": call,
                    "dependencies": [i] if i > 0 else []
                })
        
        # Format as chat
        system_msg = {
            "role": "system",
            "content": f"""You are an AI orchestrator. Select and chain models based on their metadata.

Available Models:
{json.dumps(available_models, indent=2)}

Rules:
1. Reference models by exact model_id
2. Arguments must match input_schema types
3. Set dependencies when step requires previous output
4. Parallel calls = same step, empty dependencies

Output valid JSON execution plan."""
        }
        
        dataset.append({
            "messages": [
                system_msg,
                {"role": "user", "content": query},
                {"role": "assistant", "content": json.dumps(plan, indent=2)}
            ]
        })
    
    return Dataset.from_list(dataset)

# ==================== TRAINING SETUP ====================
MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"
OUTPUT_DIR = "/kaggle/working/orchestrator-phi3-dynamic"

def formatting_func(example):
    """Format examples for training"""
    text = tokenizer.apply_chat_template(
        example["messages"], 
        tokenize=False, 
        add_generation_prompt=False
    )
    return text

# Generate data
print("Generating synthetic dataset...")
dataset = generate_orchestrator_dataset(3000)  # Reduced for faster Kaggle training
dataset = dataset.train_test_split(test_size=0.1)
print(f"Generated {len(dataset['train'])} training examples")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Quantization config - T4 optimized
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 uses fp16
    bnb_4bit_use_double_quant=True
)

# Load model - device_map="auto" handles multi-GPU automatically
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",  # Automatically splits across 2 T4s
    trust_remote_code=True,
    torch_dtype=torch.float16
)

# Prepare for training
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Training arguments for 2x T4
# Effective batch size = per_device_batch_size * gradient_accumulation * num_gpus
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,      # 2 per GPU
    gradient_accumulation_steps=4,      # Effective batch = 2*2*4 = 16
    learning_rate=1e-4,
    fp16=True,                          # T4 optimized
    bf16=False,                         # T4 doesn't support bf16
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    group_by_length=True,
    report_to="none",
    load_best_model_at_end=True,
    remove_unused_columns=False,        # Important for SFTTrainer
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_args,
    formatting_func=formatting_func,
    max_seq_length=1024,
    packing=False,
)

# Train
print("Starting training...")
trainer.train()

# Save
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

# Save a sample of training data for verification
with open(f"{OUTPUT_DIR}/sample_data.json", "w") as f:
    json.dump(dataset["train"][0]["messages"], f, indent=2)

GPUs available: 2
  GPU 0: Tesla T4
  GPU 1: Tesla T4
Generating synthetic dataset...
Generated 2700 training examples
Loading model...


ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`